In [1]:
import fitz  # PyMuPDF
import os
import re
import string
import spacy
import json
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd
import sqlite3


nlp = spacy.load("en_core_web_sm")

/global/home/users/ikolaja/.local/lib/python3.11/site-packages/torch/cuda/__init__.py:1061: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
/global/home/users/ikolaja/.local/lib/python3.11/site-packages/torch/cuda/__init__.py:1113: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12030). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  r = torch._C._cuda_getDeviceCount() if nvml_count < 0 else nvml_count


In [2]:
# Configuration
template_file = "template_sentences.json"
compute_template_embeddings = True

In [3]:
def extract_pdf_text(path):
    doc = fitz.open(path)
    pages = []
    for page in doc:
        pages.append(page.get_text("text"))
    return "\n".join(pages)

def normalize_text(text: str) -> str:
    text = text.replace("\x00", " ")
    text = re.sub(r"-\n", "", text)              # de-hyphenate line breaks
    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text


def remove_references(text: str) -> str:
    # crude but useful: cut off references section
    match = re.search(r"\n\s*(references|bibliography|acknowledgments)\s*\n", text, flags=re.I)
    return text[:match.start()] if match else text


def is_bad_sentence(sent: str) -> bool:
    s = sent.strip()
    if len(s) < 25:
        return True

    # numeric/table fragments
    chars = [c for c in s if not c.isspace()]
    if not chars:
        return True

    alpha_frac = sum(c.isalpha() for c in chars) / len(chars)
    digit_frac = sum(c.isdigit() for c in chars) / len(chars)

    if alpha_frac < 0.35:
        return True
    if digit_frac > 0.45:
        return True

    # mostly punctuation/symbols
    punct_frac = sum(c in string.punctuation for c in chars) / len(chars)
    if punct_frac > 0.45:
        return True

    # reference-like fragments
    if re.match(r"^\[?\d+\]?\s+[A-Z][A-Za-z\-]+", s):
        return True

    # equation-heavy fragments
    if len(re.findall(r"[=<>±∑√∫≈×]", s)) >= 3:
        return True

    return False


def clean_sentences(sentences) -> list[str]:
    cleaned = []
    for sent in sentences:
        sent = re.sub(r"\s+", " ", sent).strip().lower()
        if not is_bad_sentence(sent):
            cleaned.append(sent)
    return cleaned

def remove_page_artifacts(text: str) -> str:
    lines = text.splitlines()
    kept = []
    for line in lines:
        l = line.strip()

        # page numbers
        if re.fullmatch(r"\d{1,4}", l):
            continue

        # very short headers/footers
        if len(l) < 5:
            continue

        # journal/page boilerplate examples
        if re.search(r"doi:|copyright|received:|accepted:", l, flags=re.I):
            continue

        kept.append(line)
    return "\n".join(kept)

# PDF tokenization pipeline

In [9]:
pdf_paths = os.listdir("pdfs")

sentence_data = {}

for file_name in pdf_paths:
    if "pdf" not in file_name:
        continue
    exfor_id = file_name.split(".")[0]
    raw_text = extract_pdf_text(f"pdfs/{file_name}")
    text = normalize_text(raw_text)
    text = remove_references(text)
    text = remove_page_artifacts(text)
    doc = nlp(text)
    sentences = [sent.text.strip() for sent in doc.sents]
    sentences = clean_sentences(sentences)
    out_file = file_name.replace(".pdf", ".json")
    sentence_data[exfor_id] = sentences
    #with open(f"tokens/{out_file}", "w") as f:
    #    json.dump(text_data, f)

In [10]:
sentence_data

{'30077': ['lviii b, n. 2 11 dicembre 1968 study of the average level spacing from neutron-capture cross-section.',
  's. s. hasa~-, a. k. ci~aube¥ and m. l. sehgal physics department, aligarh muslim university - aligarh (ricevuto il 10 giugno 1968)',
  'neutron-activation cross-sections have been measured for 19 cases at 24 kev, a sb-be photoneutron source was used for these measurements.',
  'the value of the average level spacing d has been estimated for these cases using statistical theory of nuclear reaction.',
  'theae estimated values of average level spacing have been compared with the experimental values wherever they are known from low-energy resonance parameter data.',
  'an agreement between these two sets of values is found.',
  'the (n, y) cross-sections at 24kev have been measured by many work- ers (1-~).',
  'the present work was undertaken to complete the (n, y) cross-section data at 24 kev. an intense antimony-beryllium photoneutron source along with a low-background 

## Report Embedding Pipeline

In [11]:
def get_embeddings_batch(texts: list[str], batch_size: int = 32) -> np.ndarray:
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        inputs = tokenizer(batch, return_tensors='pt', truncation=True,
                           max_length=512, padding=True)
        with torch.no_grad():
            outputs = model(**inputs)

        mask = inputs['attention_mask'].unsqueeze(-1).float()
        embeddings = (outputs.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        all_embeddings.append(embeddings.numpy())

    return np.vstack(all_embeddings)

def get_keyword_gate_weights(sentences, keyword_gates, gate_weight=0.7):
    """
    Returns one gate weight per sentence.

    If a sentence contains any keyword, weight = 1.0.
    If not, weight = gate_weight.

    Example:
        gate_weight=0.4 means non-keyword sentences are penalized to 40%.
    """
    patterns = [
        re.compile(rf"\b{re.escape(keyword.lower())}\b")
        for keyword in keyword_gates
    ]

    weights = []
    for sent in sentences:
        s = sent.lower()
        has_keyword = any(pattern.search(s) for pattern in patterns)
        weights.append(1.0 if has_keyword else gate_weight)

    return np.array(weights)

def get_max_similarity(sentences, report_embeddings, template_category_data, gate_weight=0.9, print_out=False):
    template_sentences = template_category_data["sentences"]
    template_embeddings = template_category_data["embeddings"]
    template_key_words = template_category_data["keywords"]
    
    keyword_weights = get_keyword_gate_weights(sentences, template_key_words, gate_weight)
    similarity_matrix = cosine_similarity(report_embeddings, template_embeddings)
    weighted_similarity_matrix = keyword_weights[:, None]*similarity_matrix
    
    max_sim = np.max(weighted_similarity_matrix)
    closest_sentence = sentences[np.argmax(np.max(weighted_similarity_matrix,1))]

    if print_out:
        for i in range(len(sentences)):
            for j in range(len(template_sentences)):
                print(f"Similarity({i}, {j}): {weighted_similarity_matrix[i][j]:.4f}")
                if len(sentences[i]) > 70:
                    print(f"  \"{sentences[i][:70]}...\"")
                else:
                    print(f"  \"{sentences[i]}.\"")
                if len(template_sentences[j]) > 70:
                    print(f"  \"{template_sentences[j][:70]}...\"")
                else:
                    print(f"  \"{template_sentences[j]}.\"")
                print()
    return max_sim, closest_sentence

In [12]:
# Load SciBERT
tokenizer = AutoTokenizer.from_pretrained('allenai/scibert_scivocab_uncased')
model = AutoModel.from_pretrained('allenai/scibert_scivocab_uncased')
model.eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(31090, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [13]:
with open(template_file, "r") as f:
    template_data = json.load(f)
if compute_template_embeddings:
    for key in template_data:
        template_data[key]["embeddings"] = get_embeddings_batch(template_data[key]["sentences"])

In [14]:
file_name

'13741.pdf'

In [30]:
report_paths = os.listdir("tokens")
similarity_dataset = []
embedding_dataset = []

for exfor_id in sentence_data:
    #exfor_id = file_name.split(".")[0]
    #if "json" not in file_name:
    #    continue
    #with open(f"tokens/{file_name}", "r") as f:
    #    report_data = json.load(f)
    report_sentences = sentence_data[exfor_id] 
    if len(report_sentences) == 0:
        print(f"#{exfor_id} report empty.")
        continue
    report_embeddings = get_embeddings_batch(report_sentences) 

    # Compute similarity scores for uncertainty imputation
    report_data = {"EXFOR_Entry":exfor_id}
    for key in template_data:
        max_sim, closest_sentence = get_max_similarity(report_sentences, 
                       report_embeddings, 
                       template_data[key], 
                       print_out=False)
        report_data[f"{key}_max_sim"] = max_sim
        #report_data[f"{key}_match"] = closest_sentence
    report_data["mean_embedding"] = np.mean(report_embeddings, axis=0)
    similarity_dataset.append(report_data)


    # Create sentence wise dataframe
    report_sentence_df = pd.DataFrame({"Text":report_sentences, 
                                       "Sentence_Number": np.arange(1,1+len(report_sentences) ),
                                       "Embedding": [embedding.tobytes() for embedding in report_embeddings]})
    report_sentence_df["EXFOR_Entry"] = exfor_id
    embedding_dataset.append(report_sentence_df)


#20115 report empty.
#11601 report empty.
#11913 report empty.
#23313 report empty.
#12278 report empty.


In [33]:
similarity_df = pd.DataFrame(similarity_dataset)
similarity_df['mean_embedding'] = similarity_df['mean_embedding'].apply(lambda x: x.tobytes())

In [34]:
similarity_df

,EXFOR_Entry,background_treatment_max_sim,detector_efficiency_max_sim,normalization_treatment_max_sim,sample_attenuation_scattering_corrections_max_sim,dead_time_max_sim,coincidence_max_sim,statistical_uncertainty_max_sim,uncertainty_propagation_max_sim,time_of_flight_max_sim,mean_embedding
0,30077,0.763860,0.723606,0.728528,0.730415,0.681700,0.688365,0.718977,0.672404,0.745685,b'G\x1b\xd4<[\x9d\xc0\xbd\xeeR\x07=\x10\xad\x8...
1,22580,0.720586,0.774651,0.718130,0.729046,0.691162,0.661350,0.732901,0.683522,0.739081,b'\xe5p\x03;3\x9f\x10\xbex\xfc\xbd<\x8a\x16*>!...
2,12751,0.802526,0.795518,0.771370,0.822537,0.716145,0.698588,0.785244,0.749987,0.821927,"b'\x10\x12\xa5=,\n#\xbc\xdd:\x07>\x17K\x8c=7A\..."
3,30248,0.717319,0.728415,0.800344,0.815406,0.702081,0.685852,0.734408,0.697086,0.726499,"b'\x1c""\xc5=\xb6\x0b\xd0\xbd\x93\xc2\xc5\xbd\x..."
4,13907,0.796960,0.812552,0.817262,0.853923,0.796934,0.741350,0.696525,0.703742,0.735059,b'\x0b\x18^=>\x8d\x03\xbei=1=\xc6\xac\xbc=X^\x...
5,22838,0.742620,0.788468,0.803078,0.750314,0.784167,0.772738,0.751077,0.717163,0.738665,b'\x18\xc1\xa8\xbc\xc6\xc5z\xbd\xe1\xcf\x8c;\x...
6,10982,0.732478,0.722898,0.809790,0.756521,0.707833,0.684568,0.766369,0.720734,0.759836,b'\xd3A\xd0<B!\xd0=b\n\x92\xbd\x97\x9d\xe3=o`\...
7,10047,0.811235,0.782851,0.768611,0.802043,0.698535,0.675585,0.768928,0.694617,0.779741,b'\x96\xc3/=}\x13\xb1\xbdR\x13\xe4=\xc8.\xae=\...
8,13753,0.796146,0.816812,0.809220,0.845416,0.776881,0.764618,0.774390,0.769417,0.741937,b'\xc4$\x16\xbdQ\x8d\xaf\xbd\xff:\x10=0\xdd\xd...
9,20638,0.736500,0.783248,0.784646,0.828442,0.690464,0.678830,0.722786,0.765866,0.752490,"b'J\xe9\xf6=]\x1c\xcb\xbc\xb0""\x92=\xb3\x96\xb..."


In [37]:
embedding_df = pd.concat(embedding_dataset, ignore_index = True)
embedding_df

,Text,Sentence_Number,Embedding,EXFOR_Entry
0,"lviii b, n. 2 11 dicembre 1968 study of the av...",1,b'\x80_\xcd\xbc\x84\xc0?\xbe\xd7\n%\xbe\r\xc0=...,30077
1,"s. s. hasa~-, a. k. ci~aube¥ and m. l. sehgal ...",2,b'f\x8bK=\xc4\x0f\x9b>PT\xe5>J\xb4p>\x00\x89r\...,30077
2,neutron-activation cross-sections have been me...,3,b'\x10\x18\xa1\xbeht\xf1\xbd\x04\x18\x12\xbe\x...,30077
3,the value of the average level spacing d has b...,4,b'9\xcf\xb1\xbd\xdcJ\x01=\x1am\xaa\xbd\x19\xec...,30077
4,theae estimated values of average level spacin...,5,b'\x82\xecL\xbe&Q\xe9\xbd\xcd\x18\x98\xbe\xb4\...,30077
...,...,...,...,...
7188,thus it appears that the valence model can acc...,87,"b'*\xb6""\xbe\xb9\x07\x02\xbex\xd5\x1f\xbem$U=\...",13741
7189,"however, since the observed correlation coeffi...",88,b'\xb0-\xa0>|\xa6+\xbep\'x<\xd6\x03\x82<\xf1\x...,13741
7190,in 93nb there is evidence that this other comp...,89,b'\xc5\x06\xec=\xba\x94~\xbe~j\x90\xbe{\xe3\x0...,13741
7191,and so there is considerable interest in obtai...,90,b'y\x00D>\xf9x\xb4\xbe\xe30\x8b\xbdS\xf5\xad>\...,13741


In [38]:
conn = sqlite3.connect('output/nugrade_data.db')
measurement_df = pd.read_sql('SELECT * FROM measurements', conn)


In [39]:
similarity_df.to_sql('report_embeddings', conn, if_exists='replace', index=False)
embedding_df.to_sql('sentence_embeddings', conn, if_exists='replace', index=False)

7193

In [40]:
conn.close()